# Interactive TS analysis across a series with `pipeline_v2`

This notebook shows how to use `citkid.pipeline_v2.interactive.run_ts_series` to run the standard timestream (`'ts'`) analysis across a series of measurements, such as a series in power, temperature, or field, and to compare its results across the series.

It works like `run_iq_series` (see `2_iq-series.ipynb`), but each series point runs the `'ts'` analysis instead of IQ fitting:

1. **Gain fit** (`fit_gain`)
2. **IQ circle fit** (`fit_iq_circle`, `get_idx_t`, `get_theta_phase_offset`)
3. **x calibration** (`get_xcal_mask`, `fit_x_theta`)

The left-hand series plot shows any scalar output of the calibration and analysis on x and y, chosen from two dropdowns in the toolbar, for example the reduced PSD `sxx_10` against power `ares`.

---

## What `run_ts_series` expects

1. A function `make_custom_steps(series_idx)` that returns the calibration loading steps for one series point, including the timestream data.
2. An output zarr root. Series point `i` is stored in `root/series_000`, `root/series_001`, ...
3. The number of series points, `n_series`.

The calibration and analysis YAMLs default to the built-in `'ts'` templates.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline_v2.framework import plStep
from citkid.pipeline_v2.interactive import run_ts_series

# ---------------------------------------------------------------------
# Raw input and output stores
# ---------------------------------------------------------------------

# Raw data source (read-only)
root_raw = zarr.open(
    'path/to/raw_data.zarr',
    mode='r',
)

# Output root. run_ts_series creates/uses subgroups like
# series_000, series_001, ... inside this store.
root_out = zarr.open_group(
    'path/to/output_ts_series.zarr',
    mode='a',
)

# Number of series points and resonators
n_series = ...   # int
nrows    = ...   # int

## Define the per-series calibration loading steps

`make_custom_steps(series_idx)` returns the calibration steps for exactly one series point. Compared to the IQ series, it must also load the timestream data. These steps match `pipeline_v2/templates/custom_steps_template.py`:

| step | outputs | `func_type` |
|---|---|---|
| `load_global_data` | `fres_all`, `qres_all`, `dt`, `nrows` | `global` |
| `load_global_res_data` | `fres`, `qres`, `ares`, `res_idxs` | `global-res` |
| `load_ft` | `ft` (tone frequency) | `vectorized` |
| `load_zt` | `zt` (complex timestream) | `vectorized` |
| `load_data_f` | fine sweep `ff`, `zf` | `per-row` |
| `load_data_g` | gain sweep `fg`, `zg` | `per-row` |

`dt` is the timestream sample spacing in seconds. Sweeps must be sorted by frequency.

In [ ]:
def make_custom_steps(series_idx):
    """Return the calibration-loading steps for one series index."""

    def load_global_data():
        # Values shared across all resonators for this series index
        fres_all = ...
        qres_all = ...
        dt = ...        # timestream sample spacing (s)
        return fres_all, qres_all, dt, nrows

    def load_global_res_data():
        # One value per resonator, computed once for all rows
        fres = ...
        qres = ...
        ares = ...      # often the x-axis value for the series plot
        res_idxs = ...
        return fres, qres, ares, res_idxs

    def load_ft(data_idx):
        # Tone frequencies for the requested rows (vectorized)
        ft = ...
        return ft

    def load_zt(data_idx):
        # Complex timestreams for the requested rows (vectorized), shape (len(data_idx), n_samples)
        zt = ...
        return zt

    def load_data_f(data_idx):
        # Fine sweep for one resonator. Frequencies should be ascending.
        ff = ...
        zf = ...
        return ff, zf

    def load_data_g(data_idx):
        # Gain sweep for one resonator. Frequencies should be ascending.
        fg = ...
        zg = ...
        return fg, zg

    return [
        plStep('load_global_data', load_global_data, [], ['fres_all', 'qres_all', 'dt', 'nrows'], 'global'),
        plStep('load_global_res_data', load_global_res_data, [], ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
        plStep('load_ft', load_ft, ['data_idx'], ['ft'], 'vectorized'),
        plStep('load_zt', load_zt, ['data_idx'], ['zt'], 'vectorized'),
        plStep('load_data_f', load_data_f, ['data_idx'], ['ff', 'zf'], 'per-row'),
        plStep('load_data_g', load_data_g, ['data_idx'], ['fg', 'zg'], 'per-row'),
    ]

## Choose what the series plot shows

`x` and `y` set the initial quantities; you can change them at any time with the **x:** and **y:** dropdowns in the toolbar. By default the dropdowns offer the scalar outputs of the `'ts'` calibration and analysis:

- `ares`, `fres`, `qres`, `ft`
- `circ_radius`, `theta_phase_offset`, `idx_t`
- the reduced PSD `sxx_0p1`, `sxx_0p3`, `sxx_1`, `sxx_3`, `sxx_10`, `sxx_30`, `sxx_100`, `sxx_300` (mean S_xx within 20% of 0.1 ... 300 Hz)
- `sfactor_0p1` ... `sfactor_300` (mean of spar - sper within 20% of each frequency)

To offer other quantities, pass `quantities`: a list of parameter names, or a dict mapping a label to a parameter name or a function `f(AR, data_idx)` for derived values. Reduced PSDs are tiny numbers, so scaling them can make the plot easier to read.

Computing a reduced PSD takes a full PSD of each timestream, which is slow. So the background worker computes the selected x and y for every resonator right after pre-fitting it. When you pick another quantity, it is computed for every resonator in the background, starting after the current one. Values already computed are kept, so switching back is instant.

In [ ]:
# Optional: custom quantities. None uses the defaults listed above.
quantities = {
    'Power (dBm)': 'ares',
    'Sxx(10 Hz) (1/Hz)': 'sxx_10',
    'Sxx(1 Hz) (1/Hz)': 'sxx_1',
    'log10 Sxx(10 Hz)': lambda AR, data_idx: np.log10(AR.DS.sxx_10[data_idx]),
    'sfactor(10 Hz) (dB)': 'sfactor_10',
}

## Optional: fit y vs x for each resonator

As in the IQ series, pass a `SeriesXYFit` to fit the plotted y vs x, one fit per resonator. The fit always uses whatever is currently plotted, so it is redone when you change the dropdowns. `fit(x, y)` gets the plotted points that have both x and y, sorted by x. If you want a log scale, take the log inside `fit` and `model`, or plot a log quantity like `'log10 Sxx(10 Hz)'` above.

The outputs are shown in the series plot title and saved to `root_out['xy_fit']` (or `group=` if you pass one).

In [ ]:
from citkid.pipeline_v2.interactive import SeriesXYFit

def fit_line(x, y):
    slope, intercept = np.polyfit(x, y, 1)
    return slope, intercept

xy_fit = SeriesXYFit(
    fit = fit_line,
    output_names = ['slope', 'intercept'],
    model = lambda xs, slope, intercept: slope * xs + intercept,
    name = 'line',
)

### Optional: use DataSets you already have

Instead of `make_custom_steps`, you can pass `datasets`: one existing `DataSet` per series point, in order, all with the same number of rows. Their stored results are used and extended, wherever they live. In that case `n_series` is `len(datasets)`, and `root` is optional: it only holds the session state (for resuming) and the xy fits, so leave it out if you don't need to resume.

The background worker opens its own copies of the datasets (`DataSet.copy()`), so they must be reopenable: either built with `custom_path` / an embedded definition, or with `custom_cal_steps` in this session. While the window is open, every dataset writes through its fast write buffer; the buffer is merged and each dataset's own `write_buffer` setting is restored when the window closes.

If the DataSets don't store the x quantity but each one has a single known value (e.g. each DataSet is one drive power), pass `x_values=[...]`, one number per series point, and `x_name` for its label. It is added to the dropdowns and selected as x (unless you pass `x`). In `quantities`, `SeriesValues([...])` does the same for any other per-series-point quantity.

In [ ]:
from citkid.pipeline_v2 import DataSet

# For example, one DataSet per series point in separate zarr files:
# datasets = [
#     DataSet(zarr_path=f'path/to/point_{i}.zarr', cal_yaml_path='ts', custom_path='custom_steps.py')
#     for i in range(n_series)
# ]
# run_ts_series(
#     datasets = datasets,
#     root = zarr.open_group('path/to/series_state.zarr', mode='a'),   # optional
#     ...,                                                              # other arguments as below
# )

In [ ]:
run_ts_series(
    make_custom_steps = make_custom_steps,
    root = root_out,
    n_series = n_series,
    x = 'Power (dBm)',          # labels from `quantities` (or default names such as 'ares')
    y = 'log10 Sxx(10 Hz)',
    quantities = quantities,    # None = default 'ts' quantities
    start_idx = None,           # None = first data_idx not yet viewed; or a position in data_idxs
    # data_idxs = [0, 3, 7],    # optional subset of rows; None = all rows
    # xy_fit = xy_fit,          # optional y vs x fit per resonator (see above)
    # x_values = [...],         # one fixed x per series point (label: x_name)
    # ui_scale = 0.8,
    # plot_scale = 0.6,
)

## Practical notes

- Navigation and shortcuts are the same as in the IQ series window: **A/D** change resonator, **W/S** change series point, **N** runs panel N and all following panels (**Shift+N** only panel N), **B** marks the selected series point bad, **Shift+B** marks all series points of the resonator bad, and **Shift+A** applies the current panel settings to every series point.
- Click a point on the series plot (or use **W/S**) to load that series point into the panels on the right.
- Pre-fitting runs in the background: every resonator in `data_idxs` is fitted for every series point, starting after the current one, and then its selected x and y are computed. A resonator whose fit failed is not retried automatically; rerun its panels by hand.
- The window saves panel outputs when you explicitly save, when you change selection, or when the window closes.
- Re-running an earlier panel marks later panels stale for the current selection, and the window asks before you leave that stale state.

## Stopping and resuming

- **Finish checking the open resonator before you close the window.** A `data_idx` counts as *viewed* when you move to another resonator or close the window while it is open.
- With `start_idx = None`, the window opens at the first entry of `data_idxs` you haven't viewed.
- Fits saved in earlier sessions are loaded, not rerun. The plotted values aren't saved, so after reopening they are computed again in the background.
- The progress is stored in `root.attrs['series_state']`. To start over, delete it: `del root_out.attrs['series_state']`.

### Storage

During a session, results are written to a fast buffer with one small file per resonator. When you close the window it waits for the current background work, then merges the buffer into the compact (sharded) arrays. Close the window normally before copying the zarr to another computer. If Python crashes before closing, nothing is lost: the data is still readable, and it is merged the next time you close the window.